In [1]:
import numpy as np
import pandas
import pm4py
from matplotlib import pyplot as plt
from sklearn.mixture import GaussianMixture
import scipy.stats as stats
import ot
import os
from tqdm import tqdm
import collections
import matplotlib.dates as md
import importlib
import pickle
import random
import CRPS.CRPS as pscore


pandas.set_option('display.max_columns', None)
#pandas.set_option('display.max_rows', None)


import sys
sys.path.append('../../TaskExecutionTimeMining/')
from drbart_parser import *
from event_log_transformer import *

#sys.path.append('../../Evaluation')
sys.path.append('../../Evaluation/')
from normal_evaluation.dumas_evaluation import *

In [2]:
with open('./dumas_model.pickle', 'rb') as f:
    dumas_model = pickle.load(f)

with open('./dumas_model_no_resource.pickle', 'rb') as f:
    dumas_model_no_resource = pickle.load(f)

In [3]:
import conduct_evaluation
get_pscores = lambda likelihoods : [pscore(likelihoods[1][i], likelihoods[2][k][3]).compute()[0] for i, k in enumerate(list(likelihoods[0].keys()))]
n_processes = 12

N = 1000

In [4]:
with open('../transformed_event_logs/BPIC_2017_all_train.pickle', 'rb') as f:
    train_data = pickle.load(f)

with open('../transformed_event_logs/BPIC_2017_all_test.pickle', 'rb') as f:
    test_data = pickle.load(f)

/tmp/ipykernel_2308/3084422296.py:2: DeprecationWarning: numpy.core.numeric is deprecated and has been renamed to numpy._core.numeric. The numpy._core namespace contains private NumPy internals and its use is discouraged, as NumPy internals can change without warning in any release. In practice, most real-world usage of numpy.core is to access functionality in the public NumPy API. If that is the case, use the public NumPy API. If not, you are using NumPy internals. If you would still like to access an internal attribute, use numpy._core.numeric._frombuffer.
  train_data = pickle.load(f)


/tmp/ipykernel_2308/3084422296.py:5: DeprecationWarning: numpy.core.numeric is deprecated and has been renamed to numpy._core.numeric. The numpy._core namespace contains private NumPy internals and its use is discouraged, as NumPy internals can change without warning in any release. In practice, most real-world usage of numpy.core is to access functionality in the public NumPy API. If that is the case, use the public NumPy API. If not, you are using NumPy internals. If you would still like to access an internal attribute, use numpy._core.numeric._frombuffer.
  test_data = pickle.load(f)


In [5]:
evaluator = conduct_evaluation.ConductEvaluation(dumas_model, SampleOutcomes_Dumas_Normal,
                                                 {
                                                        'activity_key' : 'concept:name',
                                                        'resource_key' : 'org:resource_start',
                                                 },
                                     train_data, n=N, n_processes = n_processes)
likelihoods_train_A_R = evaluator.sample_cases(plot_cases=False, multiprocessing=True)

  0%|                                                                  | 0/24591 [00:00<?, ?it/s]

  0%|                                                      | 1/24591 [00:21<146:46:05, 21.49s/it]

  2%|█▏                                                      | 501/24591 [00:26<16:05, 24.95it/s]

  4%|██▏                                                    | 1001/24591 [00:30<08:31, 46.14it/s]

  6%|███▎                                                   | 1501/24591 [00:35<06:10, 62.32it/s]

...

 94%|█████████████████████████████████████████████████▌   | 23001/24591 [03:40<00:05, 268.91it/s]

 96%|██████████████████████████████████████████████████▋  | 23501/24591 [03:41<00:03, 331.19it/s]

 96%|██████████████████████████████████████████████████▋  | 23501/24591 [03:53<00:03, 331.19it/s]

 98%|████████████████████████████████████████████████████▋ | 24001/24591 [04:15<00:13, 43.58it/s]

100%|██████████████████████████████████████████████████████| 24591/24591 [04:15<00:00, 96.07it/s]

In [6]:
np.mean([v.ln() for v in likelihoods_train_A_R[0].values()])

Decimal('-4.218567886842253414761506441')

In [7]:
np.mean(get_pscores(likelihoods_train_A_R))

np.float64(833515.8160252117)

In [8]:
evaluator = conduct_evaluation.ConductEvaluation(dumas_model, SampleOutcomes_Dumas_Normal,
                                                 {
                                                        'activity_key' : 'concept:name',
                                                        'resource_key' : 'org:resource_start',
                                                 },
                                     test_data, n=N, n_processes = n_processes)
likelihoods_test_A_R = evaluator.sample_cases(plot_cases=False, multiprocessing=True)

  0%|                                                                   | 0/6018 [00:00<?, ?it/s]

  0%|                                                         | 1/6018 [00:05<9:36:01,  5.74s/it]

  8%|████▋                                                    | 501/6018 [00:07<01:07, 81.78it/s]

 17%|█████████▏                                             | 1001/6018 [00:08<00:31, 160.71it/s]

 25%|█████████████▋                                         | 1501/6018 [00:10<00:20, 224.96it/s]

...

 66%|████████████████████████████████████▌                  | 4001/6018 [01:04<00:08, 233.58it/s]

 75%|█████████████████████████████████████████▏             | 4501/6018 [01:07<00:06, 218.44it/s]

 83%|█████████████████████████████████████████████▋         | 5001/6018 [01:08<00:04, 236.43it/s]

 91%|██████████████████████████████████████████████████▎    | 5501/6018 [01:09<00:01, 291.42it/s]

100%|████████████████████████████████████████████████████████| 6018/6018 [01:09<00:00, 86.52it/s]

In [9]:
np.mean([v.ln() for v in likelihoods_test_A_R[0].values()])

Decimal('-4.584696333925458055963909693')

In [10]:
np.mean(get_pscores(likelihoods_test_A_R))

np.float64(832609.2829241164)

In [11]:
evaluator = conduct_evaluation.ConductEvaluation(dumas_model_no_resource, SampleOutcomes_Dumas_Normal,
                                                 {
                                                        'activity_key' : 'concept:name',
                                                        'resource_key' : 'org:resource_start',
                                                 },
                                     train_data, n=N, n_processes = n_processes)
likelihoods_train_A = evaluator.sample_cases(plot_cases=False, multiprocessing=True)

  0%|                                                                  | 0/24591 [00:00<?, ?it/s]

  0%|                                                      | 1/24591 [00:24<167:02:24, 24.45s/it]

  2%|█▏                                                      | 501/24591 [00:30<18:28, 21.73it/s]

  4%|██▏                                                    | 1001/24591 [00:30<07:33, 52.03it/s]

  6%|███▎                                                   | 1501/24591 [00:40<07:20, 52.46it/s]

...

 92%|████████████████████████████████████████████████▍    | 22501/24591 [03:39<00:08, 251.85it/s]

 96%|██████████████████████████████████████████████████▋  | 23501/24591 [03:41<00:03, 356.19it/s]

 96%|██████████████████████████████████████████████████▋  | 23501/24591 [03:52<00:03, 356.19it/s]

 98%|████████████████████████████████████████████████████▋ | 24001/24591 [04:16<00:10, 54.45it/s]

100%|██████████████████████████████████████████████████████| 24591/24591 [04:16<00:00, 95.85it/s]

In [12]:
np.mean([v.ln() for v in likelihoods_train_A[0].values()])

Decimal('-4.303446655415779538817056370')

In [13]:
np.mean(get_pscores(likelihoods_train_A))

np.float64(833439.1289691485)

In [14]:
evaluator = conduct_evaluation.ConductEvaluation(dumas_model_no_resource, SampleOutcomes_Dumas_Normal,
                                                 {
                                                        'activity_key' : 'concept:name',
                                                        'resource_key' : 'org:resource_start',
                                                 },
                                     test_data, n=N, n_processes = n_processes)
likelihoods_test_A = evaluator.sample_cases(plot_cases=False, multiprocessing=True)

  0%|                                                                   | 0/6018 [00:00<?, ?it/s]

  0%|                                                        | 1/6018 [00:06<11:35:46,  6.94s/it]

  8%|████▋                                                    | 501/6018 [00:08<01:06, 82.94it/s]

 17%|█████████▏                                             | 1001/6018 [00:09<00:31, 157.51it/s]

 25%|█████████████▋                                         | 1501/6018 [00:09<00:16, 277.45it/s]

...

 58%|███████████████████████████████▉                       | 3501/6018 [01:03<00:14, 168.48it/s]

 66%|████████████████████████████████████▌                  | 4001/6018 [01:06<00:12, 167.84it/s]

 75%|█████████████████████████████████████████▏             | 4501/6018 [01:06<00:06, 229.81it/s]

 91%|██████████████████████████████████████████████████▎    | 5501/6018 [01:06<00:01, 406.55it/s]

100%|████████████████████████████████████████████████████████| 6018/6018 [01:06<00:00, 90.08it/s]

In [15]:
np.mean([v.ln() for v in likelihoods_test_A[0].values()])

Decimal('-4.445651074953516183537014503')

In [16]:
np.mean(get_pscores(likelihoods_test_A))

np.float64(831201.5305285468)

In [17]:
results = {
    'dumas_test_A' : likelihoods_test_A,
    'dumas_test_A_R' : likelihoods_test_A_R,
    'dumas_train_A' : likelihoods_train_A,
    'dumas_train_A_R' : likelihoods_train_A_R
}

with open('./artificial_dumas_evaluation_.pickle', 'wb') as handle:
    pickle.dump(results, handle)